In [ ]:
!pip install grpcio grpcio-tools transformers torch bitsandbytes unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/1

In [ ]:
%%writefile server.py
import grpc
from concurrent import futures
import torch
from unsloth import FastLanguageModel
import intent_service_pb2
import intent_service_pb2_grpc

MODEL_PATH = "/content/drive/MyDrive/model"

INTENT_LABELS = [
    "activate_my_card", "age_limit", "card_arrival", "change_pin",
    "exchange_rate", "lost_or_stolen_card", "passcode_forgotten",
    "request_refund", "terminate_account", "transfer_timing"
]

class IntentServiceHandler(intent_service_pb2_grpc.IntentServiceServicer):
    def __init__(self):
        print("Loading Unsloth model and adapter...")
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name = MODEL_PATH,
            max_seq_length = 2048,
            load_in_4bit = True,
        )
        FastLanguageModel.for_inference(self.model)
        print("Model loaded successfully.")

    def IntentRecognizer(self, request, context):
        prompt = f"Customer message: {request.message}\nIntent:"

        inputs = self.tokenizer([prompt], return_tensors = "pt").to("cuda")

        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens = 20)
            result = self.tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

        print(f"Predicted intent: {result}")
        predicted_intent = "unknown"
        for label in INTENT_LABELS:
            if label in result.lower():
                predicted_intent = label
                break

        return intent_service_pb2.IntentResponse(
            intent=predicted_intent,
            confidence=1.0,
            reason="Prediction completed via Unsloth"
        )

def serve():
    server = grpc.server(futures.ThreadPoolExecutor(max_workers=10))
    intent_service_pb2_grpc.add_IntentServiceServicer_to_server(IntentServiceHandler(), server)
    server.add_insecure_port('[::]:50051')
    print("Intent Service is running on port 50051...")
    server.start()
    server.wait_for_termination()

if __name__ == '__main__':
    serve()

Overwriting server.py


In [ ]:
!python -m grpc_tools.protoc -I. \
--python_out=. \
--grpc_python_out=. \
intent_service.proto

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import time

# 1. Chạy server ngầm
os.system("python server.py &")

print("Đang nạp mô hình Llama-3/Unsloth vào GPU...")
time.sleep(20)

# 2. Mở đường hầm TCP qua Pinggy (Dùng tiền tố tcp@)
!ssh -o "StrictHostKeyChecking=no" -p 443 -R0:localhost:50051 tcp@a.pinggy.io

Đang nạp mô hình Llama-3/Unsloth vào GPU...
Allocated port 8 for remote forward to localhost:50051
7=)0]8;;\                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         ┌────────────────────────────┐                                                  │                            │                                                  │ Wait while we prepare the  │                                